# 1

In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "sentence-transformers/multi-qa-mpnet-base-cos-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [4]:
from time import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
sample_text = "This is a sample text for embedding generation."
inputs = tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
times = dict()

def measure_inference_time(inputs, model, num_runs=100):
    start_time = time()
    for _ in range(num_runs):
        model(**inputs).last_hidden_state
    end_time = time()
    return (end_time - start_time) / num_runs

times["just_pytorch"] = measure_inference_time(inputs, model)

model.eval()
times["torch_eval"] = measure_inference_time(inputs, model)

with torch.no_grad():
    times["torch_no_grad"] = measure_inference_time(inputs, model)

with torch.inference_mode():
    times["torch_inference_mode"] = measure_inference_time(inputs, model)

for key, value in times.items():
    print(f"{key}: {value:.6f} seconds")

base_time = times["just_pytorch"]
for key in times:
    if key != "just_pytorch":
        speedup = base_time / times[key]
        print(f"Speedup for {key}: {speedup:.2f}x")

just_pytorch: 0.058544 seconds
torch_eval: 0.007164 seconds
torch_no_grad: 0.005678 seconds
torch_inference_mode: 0.005001 seconds
Speedup for torch_eval: 8.17x
Speedup for torch_no_grad: 10.31x
Speedup for torch_inference_mode: 11.71x


# 2

In [5]:
model.eval()
compiled_model = torch.compile(model)

start = time()
compiled_model(**inputs).last_hidden_state
end = time()
print(f"Compilation time with warmup: {(end - start):.2f} seconds")

times['with_compilation'] = measure_inference_time(inputs, compiled_model)
print('Inference time with compilation:', times['with_compilation'], 'seconds.')

base_time = times["just_pytorch"]
for key in times:
    if key != "just_pytorch":
        speedup = base_time / times[key]
        print(f"Speedup for {key}: {speedup:.2f}x")

Compilation time with warmup: 116.70 seconds
Inference time with compilation: 0.0036240410804748535 seconds.
Speedup for torch_eval: 8.17x
Speedup for torch_no_grad: 10.31x
Speedup for torch_inference_mode: 11.71x
Speedup for with_compilation: 16.15x


# 3

In [6]:
model.to('cpu')

model.eval()
model_quantized = torch.ao.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
print(model_quantized)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (k): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (v): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (o): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (dropout): Dropout(p=0.1, inplace=F

In [7]:
import os
os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/model.pth")
torch.save(model_quantized.state_dict(), "models/model_quantized.pth")

model_size = os.path.getsize("models/model.pth") / (1024 * 1024)
model_quantized_size = os.path.getsize("models/model_quantized.pth") / (1024 * 1024)
size_reduction = model_size / model_quantized_size
print(f"Original model size: {model_size:.2f} MB")
print(f"Quantized model size: {model_quantized_size:.2f} MB")
print(f"Size reduction: {size_reduction:.2f}x")

Original model size: 417.72 MB
Quantized model size: 173.10 MB
Size reduction: 2.41x


In [8]:
model.eval()
model_quantized.eval()
inputs_cpu = {k: v.to('cpu') for k, v in inputs.items()}

with torch.inference_mode():
    model_inference_time = measure_inference_time(inputs_cpu, model)
    model_quantized_inference_time = measure_inference_time(inputs_cpu, model_quantized)
    speedUp = model_inference_time / model_quantized_inference_time

print(f"Original model inference time: {model_inference_time:.6f} seconds")
print(f"Quantized model inference time: {model_quantized_inference_time:.6f} seconds")
print(f"Speedup: {speedUp:.2f}x")

Original model inference time: 0.127159 seconds
Quantized model inference time: 0.023684 seconds
Speedup: 5.37x


# 4

In [9]:
inputs_small = tokenizer("This is a sample text.", return_tensors="pt", padding=True, truncation=True)
inputs_medium = tokenizer("This is a sample text. " * 50, return_tensors="pt", padding=True, truncation=True)
inputs_large = tokenizer("This is a sample text. " * 200, return_tensors="pt", padding=True, truncation=True)

inputs_small = {k: v.to(device) for k, v in inputs_small.items()}
inputs_medium = {k: v.to(device) for k, v in inputs_medium.items()}
inputs_large = {k: v.to(device) for k, v in inputs_large.items()}
model.to(device)

model.eval()
compiled_model = torch.compile(model)
compiled_model.to(device)
compiled_model(**inputs_small)

compiled_model_with_maxautotune = torch.compile(model, mode="max-autotune")
compiled_model_with_maxautotune.to(device)
compiled_model_with_maxautotune(**inputs_small)

compiled_model_with_no_cuda_graphs = torch.compile(model, mode="max-autotune-no-cudagraphs")
compiled_model_with_no_cuda_graphs.to(device)
compiled_model_with_no_cuda_graphs(**inputs_small)

def compare_times(compiled_model, text=""):
    times = dict()
    times["input_small"] = measure_inference_time(inputs_small, compiled_model)
    times["input_medium"] = measure_inference_time(inputs_medium, compiled_model)
    times["input_large"] = measure_inference_time(inputs_large, compiled_model)
    print(text)
    for key, value in times.items():
        print(f"{key}: {value:.6f} seconds")

    return times

baseline_times = compare_times(compiled_model, "Compiled Model Inference Times default settings")
max_autotune_times = compare_times(compiled_model_with_maxautotune, "Compiled Model Inference Times with max-autotune")
no_cudagraphs_times = compare_times(compiled_model_with_no_cuda_graphs, "Compiled Model Inference Times with max-autotune-no-cudagraphs")

print("\nSpeedup with max-autotune over default:")
for key in baseline_times:
    speedup = baseline_times[key] / max_autotune_times[key]
    print(f"{key}: {speedup:.2f}x")

print("\nSpeedup with max-autotune-no-cudagraphs over default:")
for key in baseline_times:
    speedup = baseline_times[key] / no_cudagraphs_times[key]
    print(f"{key}: {speedup:.2f}x")

AUTOTUNE addmm(8x3072, 8x768, 768x3072)
  bias_addmm 0.0297 ms 100.0% 
  addmm 0.0338 ms 87.9% 
  triton_mm_89 0.0563 ms 52.7% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=128, BLOCK_M=16, BLOCK_N=32, B_PROLOGUE_CAST_TYPE=None, EVEN_K=True, GROUP_M=8, num_stages=5, num_warps=2
  triton_mm_93 0.0563 ms 52.7% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=128, BLOCK_M=16, BLOCK_N=64, B_PROLOGUE_CAST_TYPE=None, EVEN_K=True, GROUP_M=8, num_stages=5, num_warps=4
  triton_mm_86 0.0573 ms 51.8% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=128, BLOCK_M=16, BLOCK_N=32, B_PROLOGUE_CAST_TYPE=None, EVEN_K=True, GROUP_M=8, num_stages=2, num_warps=2
  triton_mm_88 0.0604 ms 49.2% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=16, BLOCK_N=32, B_PROLOGUE_CAST_TYPE=None, EVEN_K=True, GROUP_M=8, num_stages=5, num_warps=2
  triton_mm_92 0.0625 ms 47.5% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BLOCK_M=16, BLOCK_N=64, B_PROLOGUE_CAST_TYPE=None, EVEN_K=True, GROUP_M=8, nu

Compiled Model Inference Times default settings
input_small: 0.004050 seconds
input_medium: 0.007506 seconds
input_large: 0.012347 seconds
Compiled Model Inference Times with max-autotune
input_small: 0.011762 seconds
input_medium: 0.015486 seconds
input_large: 0.020514 seconds
Compiled Model Inference Times with max-autotune-no-cudagraphs
input_small: 0.004840 seconds
input_medium: 0.007263 seconds
input_large: 0.012016 seconds

Speedup with max-autotune over default:
input_small: 0.34x
input_medium: 0.48x
input_large: 0.60x

Speedup with max-autotune-no-cudagraphs over default:
input_small: 0.84x
input_medium: 1.03x
input_large: 1.03x


Second mode is much slower than default compile mode. The third mode is slightly faster than default mode for bigger inputs. We can observe tendency that bigger inputs benefit more from compilation optimizations in second and third modes.

# 5

In [10]:
capability = torch.cuda.get_device_capability()
print(f"CUDA device capability: {capability}")

# Tensor Cores are available on NVidia GPUs with CUDA >= 7 (e.g. Volta, Turing, Ampere, Hopper)
if capability >= (7, 0):
    print("Tensor Cores available: fast float16 supported.")
else:
    print("Tensor Cores not available: float16 may be slow or unsupported.")

CUDA device capability: (7, 0)
Tensor Cores available: fast float16 supported.


In [11]:
model_half = model.half().to('cuda')
outputs = model_half(
    inputs["input_ids"].to('cuda'),
    attention_mask=inputs["attention_mask"].to('cuda')
)

In [12]:
model_fp32 = torch.nn.Linear(10, 1)
data_fp32 = torch.randn(100, 10)
labels_fp32 = torch.randn(100, 1)

print(f"Data type of model_fp32 parameters: {model_fp32.weight.dtype}")
print(f"Data type of data_fp32: {data_fp32.dtype}")
print(f"Data type of labels_fp32: {labels_fp32.dtype}")

output_fp32 = model_fp32(data_fp32)
loss_fn = torch.nn.MSELoss()
loss_fp32 = loss_fn(output_fp32, labels_fp32)

print(f"Loss fp32: {loss_fp32.item()}")

Data type of model_fp32 parameters: torch.float32
Data type of data_fp32: torch.float32
Data type of labels_fp32: torch.float32
Loss fp32: 1.7915644645690918


In [13]:
model_fp16 = model_fp32.half()
data_fp16 = data_fp32.half()
labels_fp16 = labels_fp32.half()

print(f"Data type of model_fp16 parameters: {model_fp16.weight.dtype}")
print(f"Data type of data_fp16: {data_fp16.dtype}")
print(f"Data type of labels_fp16: {labels_fp16.dtype}")

output_fp16 = model_fp16(data_fp16)
loss_fp16 = loss_fn(output_fp16.float(), labels_fp16.float())

print(f"Loss fp16: {loss_fp16.item()}")

Data type of model_fp16 parameters: torch.float16
Data type of data_fp16: torch.float16
Data type of labels_fp16: torch.float16
Loss fp16: 1.7914879322052002


In [19]:
model.to(device)
model.eval()

inputs_gpu = {k: v.to(device) for k, v in inputs.items()}

model_fp32 = model.float()
with torch.inference_mode():
    time_fp32 = measure_inference_time(inputs_gpu, model_fp32)

model_fp16 = model.half()
with torch.inference_mode():
    time_fp16 = measure_inference_time(inputs_gpu, model_fp16)

model_amp = model.float()
def measure_inference_time_amp(inputs, model, num_runs=100):
    start_time = time()
    for _ in range(num_runs):
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            model(**inputs).last_hidden_state
    end_time = time()
    return (end_time - start_time) / num_runs

with torch.inference_mode():
    time_amp = measure_inference_time_amp(inputs_gpu, model_amp)

print(f"Full precision (fp32) inference time: {time_fp32:.6f} seconds")
print(f"Manual half-precision (fp16) inference time: {time_fp16:.6f} seconds")
print(f"Automatic mixed precision (AMP) inference time: {time_amp:.6f} seconds")

print(f"\nSpeedup with fp16 over fp32: {time_fp32 / time_fp16:.2f}x")
print(f"Speedup with AMP over fp32: {time_fp32 / time_amp:.2f}x")

Full precision (fp32) inference time: 0.005279 seconds
Manual half-precision (fp16) inference time: 0.005386 seconds
Automatic mixed precision (AMP) inference time: 0.007513 seconds

Speedup with fp16 over fp32: 0.98x
Speedup with AMP over fp32: 0.70x


In practice I would use fp32 because there is no speedup using fp16 or AMP.

# 6

In [ ]:
import onnxruntime as ort
from time import time

sample_input = tokenizer(
    "This is a sample input text for ONNX inference.",
    padding=True,
    truncation=True,
    return_tensors="np",
)

inputs_onnx = {
    "input_ids": sample_input["input_ids"],
    "attention_mask": sample_input["attention_mask"],
}

start = time()
sess_options_online = ort.SessionOptions()
sess_options_online.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session_online = ort.InferenceSession(
    "model.onnx",
    sess_options=sess_options_online,
    providers=["CPUExecutionProvider"]
)
session_online.run(None, inputs_onnx)
end = time()
cold_start_online = end - start

start = time()
sess_options_offline = ort.SessionOptions()
sess_options_offline.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
session_offline = ort.InferenceSession(
    "model_optimized.onnx",
    sess_options=sess_options_offline,
    providers=["CPUExecutionProvider"]
)
session_offline.run(None, inputs_onnx)
end = time()
cold_start_offline = end - start

print("Cold Start Times:")
print(f"Online optimization: {cold_start_online:.6f} seconds")
print(f"Offline optimization: {cold_start_offline:.6f} seconds")
print(f"Speedup: {cold_start_online / cold_start_offline:.2f}x")

def measure_onnx_inference_time(session, inputs, num_runs=100):
    start_time = time()
    for _ in range(num_runs):
        session.run(None, inputs)
    end_time = time()
    return (end_time - start_time) / num_runs

inference_time_online = measure_onnx_inference_time(session_online, inputs_onnx)
inference_time_offline = measure_onnx_inference_time(session_offline, inputs_onnx)

print("\nInference Times:")
print(f"Online optimization: {inference_time_online:.6f} seconds")
print(f"Offline optimization: {inference_time_offline:.6f} seconds")
print(f"Speedup: {inference_time_online / inference_time_offline:.2f}x")